In [23]:
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os
import cv2
import numpy as np
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import torch
import matplotlib
from sklearn.decomposition import PCA

# Directory to save model checkpoints
save_dir = './checkpoints'
os.makedirs(save_dir, exist_ok=True), 

import torch
torch.cuda.empty_cache()  # Clear unused memory

SHOW_LRP_OUTPUTS= True
VISUALIZE_PCA = False
VISUALIZE_TSNE = False
SHOW_GRADCAM_OUTPUTS = True
LOAD_MODEL_FROM_DISK = False # #  # True
LOAD_FEATURES_FROM_DISK = True #False #  # True
EXPERIMENT_TYPE = "black" # white" # "black" "mixed"
IS_SAVE_METRICS = False
MODEL_SAVING_PATH = None
batch_size = 64
learning_rate = 0.001
num_epochs = 5
#Save model after each epoch
def createFullPath(fileName, save_dir):
    os.makedirs(save_dir, exist_ok=True)
    return os.path.join(save_dir, EXPERIMENT_TYPE + " " + fileName)

def save_model(model, optimizer, avg_loss, test_accuracy, epoch, model_save_directory):
    os.makedirs("./saved_models", exist_ok=True)  # Create the directory if it doesn't exist
    checkpoint_path = os.path.join('./saved_models', f'{model_save_directory}.pth')
    torch.save({
        #'epoch': epoch , model_save_directory+ 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        #'loss': avg_loss,
        #'accuracy': test_accuracy
    }, checkpoint_path), model_save_directory

    print(f'Model saved at ./saved_models/{model_save_directory}')

In [24]:
#Active Conda Environment: condaPyProj311
conda_env = os.getenv("CONDA_DEFAULT_ENV")
print(f"dflt Conda Environment: {conda_env}")
# Get the path of the active conda environment
conda_env_path = os.getenv("CONDA_PREFIX")
if conda_env_path:
    # Extract the environment name from the path
    conda_env_name = os.path.basename(conda_env_path)
    print(f"Active Conda Environment: {conda_env_name}")
else:
    print("No active Conda environment")
from torchvision.datasets import ImageFolder
FIG_SIZE = (12,6)
output_directory = "./latest_explained_images/"
os.makedirs(output_directory, exist_ok=True)  # Create the directory if it doesn't exist
class ImageFolderWithFilenames(ImageFolder):
    def __getitem__(self, index):
        # Get the original tuple of (image, label)
        original_tuple = super().__getitem__(index)
        # Get the image path and extract the filename
        path, _ = self.samples[index]
        filename = os.path.basename(path)
        # Return image, label, and filename
        return original_tuple[0], original_tuple[1], filename

dflt Conda Environment: condaPyProj311
Active Conda Environment: condaPyProj311


In [25]:
train_loss_list = []
test_accuracy_list = []
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')  # Prints 'cuda' if GPU is available, else 'cpu'
image_size = (480, 960)  # Image size
color_channel = 3 # only grayscale, so 1 channel is enough
# Define the transformations (resize, normalize, convert to tensor)
transform = transforms.Compose([
    transforms.Resize(image_size),  # Resize images to 960x480 pixels
    #transforms.Grayscale(num_output_channels=color_channel),  # Convert to grayscale (1 channel)
    transforms.ToTensor(),  # Convert the image to PyTorch tensor
    #transforms.Normalize((0.5), (0.5))  # Normalize the image between -1 and 1
])
if (EXPERIMENT_TYPE == "white"):
    MODEL_SAVING_PATH = "white_mode"
    train_dataset = ImageFolderWithFilenames(root='/home/zubair/Downloads/CNN Data/Training Images white', transform=transform)
    test_dataset = ImageFolderWithFilenames(root='/home/zubair/Downloads/CNN Data/Test Images White', transform=transform)
elif (EXPERIMENT_TYPE == "black"):
    MODEL_SAVING_PATH = "black_model"
    train_dataset = ImageFolderWithFilenames(root='/home/zubair/Downloads/CNN Data/Training Images Black', transform=transform)
    test_dataset = ImageFolderWithFilenames(root='/home/zubair/Downloads/CNN Data/Test Images Black', transform=transform)
else: # (EXPERIMENT_TYPE == "mixed_color_model"):
    MODEL_SAVING_PATH = "mixed_model"    
    train_dataset = ImageFolderWithFilenames(root='/home/zubair/Downloads/CNN Data/Training Images Combined', transform=transform)
    test_dataset = ImageFolderWithFilenames(root='/home/zubair/Downloads/CNN Data/Test Images Combined', transform=transform)

# DataLoader (to handle batch processing)
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=True)

Using device: cuda


In [32]:
class SimpleCNNForLRP(nn.Module):
    def __init__(self):
        super(SimpleCNNForLRP, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(in_channels=color_channel, out_channels=16, kernel_size=3, stride=1, padding=1, device=device),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1, device=device),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1, device=device),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1, device=device),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, stride=1, padding=1, device=device),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(self._get_conv_output_size(image_size), 512),
            nn.ReLU(),
            
            nn.Linear(512, 1024),
            nn.ReLU(),
            
            nn.Linear(1024, len(train_dataset.classes))  # Output layer: size = num classes
        )
        self.feature_maps = None
        self.gradients = None

    def forward(self, x):
        x = self.conv_layers(x)
        self.feature_maps = x
        x = x.view(x.size(0), -1)  # Flatten the output of the convolution layers
        x = self.fc_layers(x)
        return x

    def _get_conv_output_size(self, img_size):
        with torch.no_grad():
            dummy_input = torch.ones(1, color_channel, *img_size).to(device)
            x = self.conv_layers(dummy_input)
            return x.numel()

    def backward_lrp(self, output, relevance):
        """
        Propagate the relevance backward using the epsilon rule for fully connected layers.
        """
        # Propagate relevance through the fully connected layers in reverse order
        for layer in reversed(self.fc_layers):
            if isinstance(layer, nn.Linear):
                relevance = self.lrp_propagate_fc(layer, relevance)
        return relevance

    def lrp_propagate_fc(self, layer, relevance):
        """
        Propagate relevance through a fully connected layer.
        """
        # Get the weights of the current layer (weights: (out_features, in_features))
        weights = layer.weight  # Shape: (out_features, in_features)
        bias = layer.bias
        
        # Calculate the relevance propagation using the epsilon rule
        # Normalize relevance by dividing by the sum of absolute weights (epsilon rule)
        epsilon = 1e-5
        
        # We use a combination of element-wise division and matrix multiplication to get relevance propagation
        # Relevance is backpropagated through the layer by dividing by weights' activations
        relevance = torch.matmul(relevance, weights)  # Propagate relevance through the weights
        relevance = relevance / (weights.abs().sum(dim=1, keepdim=True) + epsilon)  # Epsilon rule for normalization
        
        return relevance

    def get_activations_gradient(self):
        return self.gradients

    def get_activations(self, x):
        return self.conv_layers(x)
    
# Example usage:
modelForLRP = SimpleCNNForLRP().to(device)
image = torch.randn(1, color_channel, image_size[0], image_size[1]).to(device)  # Example input
output = modelForLRP(image)

# Assume we are interested in the relevance of the output with respect to a particular class
# Initialize relevance with ones (for simplicity)
relevance = torch.ones_like(output).to(device)

# Backpropagate relevance through the model
relevance_map = modelForLRP.backward_lrp(output, relevance)

# Visualize the LRP relevance map
plot_lrp_relevance(image, relevance_map)


RuntimeError: The size of tensor a (2) must match the size of tensor b (1024) at non-singleton dimension 0

In [27]:
def save_metrics_to_disk(train_loss_list, test_accuracy_list, train_loss_filename, train_accuracy_filename):
        np.save(createFullPath(train_accuracy_filename, 'saved_metrics'), test_accuracy_list)  # Save test_accuracy_list as numpy array

In [28]:
# Modify the training function to save loss, accuracy, and model
def train_model_ForLRP(model, train_loader, test_loader, criterion, optimizer, num_epochs, device):
    model.train()  # Set the model to training mode
    for epoch in range(num_epochs):
        running_loss = 0.0
        # Training loop
        for images, labels, filenames in train_loader:
            images, labels = images.to(device), labels.to(device)  # Move data to the GPU/CPU
            optimizer.zero_grad()  # Zero the gradients
            outputs = model(images)  # Forward pass
            loss = criterion(outputs, labels)  # Compute the loss
            loss.backward()  # Backpropagation
            optimizer.step()  # Update the weights
            running_loss += loss.item()
        # Average loss for the epoch
        avg_loss = running_loss / len(train_loader)
        train_loss_list.append(avg_loss)
        print(f'Epoch [{epoch + 1}/{num_epochs}], Train Loss: {avg_loss:.4f}')
        # Test accuracy at the end of each epoch
        test_accuracy = test_model(model, test_loader, device)
        test_accuracy_list.append(test_accuracy)
        # save_model(model, optimizer, avg_loss, test_accuracy, epoch);
    return train_loss_list, test_accuracy_list

# Modify the test function to return accuracy
def test_model(model, test_loader, device):
    model.eval()  # Set the model to evaluation mode
    correct = 0
    total = 0

    # with torch.no_grad():  # No need to track gradients for testing
    for images, labels, fileNames in test_loader:
        images, labels = images.to(device), labels.to(device)  # Move data to the GPU/CPU
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f'Accuracy of the model on the test images: {accuracy:.2f}%')
    return accuracy


In [29]:
modelForLRP = None
criterionForGradCam = None
optimizerForGradCam = None
if not LOAD_MODEL_FROM_DISK:
    modelForLRP = SimpleCNNForLRP().to(device)  # Move the model to the GPU/CPU
    criterionForGradCam = nn.CrossEntropyLoss()  # Cross entropy loss for multi-class classification
    optimizerForGradCam = optim.Adam(modelForLRP.parameters(), lr=learning_rate)
    train_loss_list, test_accuracy_list = train_model_ForLRP(modelForLRP, train_loader, test_loader, criterionForGradCam, optimizerForGradCam, num_epochs, device)
    save_model(modelForLRP, optimizerForGradCam, 0, 0, 0, "lrp_model");
    print( "model trained now")
    #plot_metrics(train_loss_list, test_accuracy_list)
    if IS_SAVE_METRICS:
        save_metrics_to_disk(train_loss_list, test_accuracy_list, "trainLoss-ForLRP", "testAccuracyForLRP")
    
elif LOAD_MODEL_FROM_DISK:
    modelForLRP = SimpleCNNForLRP()  # Ensure this matches your saved model
    checkpoint_path = f"./saved_models/lrp_model.pth"
    checkpoint = torch.load(checkpoint_path)
    modelForLRP.load_state_dict(checkpoint["model_state_dict"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    modelForLRP.to(device)
    print( "model fetched from ", )

Epoch [1/5], Train Loss: 0.5694
Accuracy of the model on the test images: 80.17%
Epoch [2/5], Train Loss: 0.2154
Accuracy of the model on the test images: 87.19%
Epoch [3/5], Train Loss: 0.1023
Accuracy of the model on the test images: 89.26%
Epoch [4/5], Train Loss: 0.0400
Accuracy of the model on the test images: 89.26%
Epoch [5/5], Train Loss: 0.0210
Accuracy of the model on the test images: 87.60%
Model saved at ./saved_models/lrp_model
model trained now


In [30]:
import matplotlib.pyplot as plt
import numpy as np

def plot_lrp_relevance(image, relevance):
    """
    Visualizes the LRP relevance map on top of the input image.
    Arguments:
    - image: The original image.
    - relevance: The relevance map obtained from LRP propagation.
    """
    # Convert the relevance map to a numpy array for visualization (if it's a torch tensor)
    relevance = relevance.squeeze().cpu().detach().numpy()

    # Normalize the relevance map for visualization
    relevance_normalized = np.maximum(relevance, 0)  # Make sure to keep only positive relevance
    relevance_normalized = (relevance_normalized - relevance_normalized.min()) / (relevance_normalized.max() - relevance_normalized.min())

    # Normalize the image for visualization
    image = image.squeeze().cpu().detach().numpy()
    image = np.transpose(image, (1, 2, 0))  # Convert to HWC format (Height, Width, Channels)

    # Plot the image and the relevance map
    plt.figure(figsize=(10, 10))
    plt.subplot(1, 2, 1)
    plt.imshow(image)
    plt.title("Original Image")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(image)
    plt.imshow(relevance_normalized, alpha=0.5, cmap='jet')  # Overlay relevance map
    plt.title("LRP Relevance Map")
    plt.axis('off')

    plt.show()


In [31]:
if (SHOW_LRP_OUTPUTS):
    import matplotlib
    %matplotlib inline
    images, labels, filenames = next(iter(train_loader))

    for image, label, fielname in zip(images, labels, filenames):
        
        image = image.unsqueeze(0).to(device)

    # Forward pass
    output = modelForLRP(image)
    
    # Assume we are interested in the relevance of the output with respect to a particular class
    relevance = modelForLRP.backward_lrp(output, torch.ones_like(output))  # Initialize relevance with ones (for simplicity)
    plot_lrp_relevance(image, relevance)

    
    # Visualize the relevance or further processing


RuntimeError: The size of tensor a (2) must match the size of tensor b (1024) at non-singleton dimension 0